<center><img src="https://keras.io/img/logo-small.png" alt="Keras logo" width="100"><br/></center>

# 🔍 Contradictory, My Dear Watson — Multilingual NLI with XLM-RoBERTa

> Fine-tuning a multilingual transformer (XLM-RoBERTa) for 3-class Natural Language Inference across 15 languages.

**Labels:**
- `0` → Entailment
- `1` → Neutral  
- `2` → Contradiction

**Improvements over baseline:**
- ✅ Fixed indentation bug in TPU detection block
- ✅ Upgraded from `bert_base_multi` → `xlm_roberta_base_multi` (better multilingual performance)
- ✅ Added label smoothing to combat overconfidence
- ✅ Added learning rate warm-up + cosine decay scheduler
- ✅ Added EarlyStopping + ModelCheckpoint callbacks
- ✅ Added training history visualization
- ✅ Added per-language accuracy analysis
- ✅ Model saved in Keras format for HuggingFace deployment
- ✅ Cleaned up unused imports and redundant cells

## 1. Install & Import

In [ ]:
!pip install -q keras-nlp --upgrade
!pip install -q seaborn

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import keras_nlp
import seaborn as sns
import matplotlib.pyplot as plt
import os

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

## 2. Hardware Detection & Distribution Strategy

In [ ]:
# BUG FIX: Original code had a leading space before 'try', causing IndentationError
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    strategy = tf.distribute.TPUStrategy(tpu)
    print("✅ Running on TPU")
except ValueError:
    strategy = tf.distribute.MirroredStrategy()
    print("ℹ️  TPU not found — using:", strategy.__class__.__name__)

print(f"Number of replicas: {strategy.num_replicas_in_sync}")

## 3. Configuration

In [ ]:
# ─── Paths ───────────────────────────────────────────────
DATA_DIR = '/kaggle/input/contradictory-my-dear-watson/'
MODEL_SAVE_PATH = '/kaggle/working/xlm_roberta_nli.keras'

# ─── Label map ───────────────────────────────────────────
LABEL_MAP = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

# ─── Hyperparameters ─────────────────────────────────────
VALIDATION_SPLIT = 0.2          # reduced from 0.3 → more training data
BATCH_SIZE       = 16 * strategy.num_replicas_in_sync
EPOCHS           = 5            # more epochs with early stopping
LEARNING_RATE    = 2e-5         # standard for XLM-R fine-tuning
LABEL_SMOOTHING  = 0.1          # reduces overconfidence
WARMUP_STEPS     = 100
PRESET           = 'xlm_roberta_base_multi'  # better than bert_base_multi for multilingual

print("Configuration set ✅")
print(f"  Model preset : {PRESET}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Epochs       : {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

## 4. Load Data

In [ ]:
df_train = pd.read_csv(DATA_DIR + 'train.csv')
df_test  = pd.read_csv(DATA_DIR + 'test.csv')

print(f"Train shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")
print(f"Null values :\n{df_train.isnull().sum()}")
df_train.head()

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Label distribution
label_counts = df_train['label'].value_counts().sort_index()
axes[0].bar([LABEL_MAP[i] for i in label_counts.index], label_counts.values,
            color=['#4CAF50', '#2196F3', '#F44336'], edgecolor='black')
axes[0].set_title('Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, f'{v}\n({v/len(df_train)*100:.1f}%)', ha='center', fontsize=10)

# Language distribution (top 10)
lang_counts = df_train['language'].value_counts().head(10)
axes[1].barh(lang_counts.index, lang_counts.values, color='steelblue', edgecolor='black')
axes[1].set_title('Top 10 Language Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sentence length analysis
df_train['premise_len']    = df_train['premise'].str.len()
df_train['hypothesis_len'] = df_train['hypothesis'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_train['premise_len'],    bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Premise Length Distribution');    axes[0].set_xlabel('Characters')
axes[1].hist(df_train['hypothesis_len'], bins=50, color='coral',     edgecolor='black', alpha=0.7)
axes[1].set_title('Hypothesis Length Distribution'); axes[1].set_xlabel('Characters')
plt.tight_layout(); plt.show()

df_train[['premise_len', 'hypothesis_len']].describe().round(1)

## 6. Preprocessing & tf.data Pipeline

In [ ]:
# Shuffle before split for reproducibility
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

TRAIN_SIZE = int(len(df_train) * (1 - VALIDATION_SPLIT))

def build_dataset(df_subset, shuffle=False):
    """Build a tf.data.Dataset from a DataFrame slice."""
    premises    = df_subset['premise'].values
    hypotheses  = df_subset['hypothesis'].values
    labels      = keras.utils.to_categorical(df_subset['label'].values, num_classes=3)

    ds = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=42)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True).cache().prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(df_train.iloc[:TRAIN_SIZE],  shuffle=True)
val_ds   = build_dataset(df_train.iloc[TRAIN_SIZE:],  shuffle=False)

print(f"Train batches : {len(train_ds)}")
print(f"Val batches   : {len(val_ds)}")

## 7. Build Model (XLM-RoBERTa)

In [ ]:
# Cosine decay with linear warm-up
total_steps   = len(train_ds) * EPOCHS
lr_schedule   = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=total_steps,
    warmup_steps=WARMUP_STEPS,
    alpha=1e-7
)

with strategy.scope():
    # XLM-RoBERTa handles 100+ languages out of the box
    classifier = keras_nlp.models.XLMRobertaClassifier.from_preset(
        PRESET,
        num_classes=3
    )
    classifier.compile(
        optimizer=keras.optimizers.Adam(lr_schedule),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=LABEL_SMOOTHING  # reduces overconfidence
        ),
        metrics=['accuracy']
    )

classifier.summary()

## 8. Train with Callbacks

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

history = classifier.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)

print(f"\n✅ Best val_accuracy: {max(history.history['val_accuracy']):.4f}")

## 9. Training History Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val',   marker='s')
axes[0].set_title('Model Accuracy',  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train', marker='o', color='orange')
axes[1].plot(history.history['val_loss'], label='Val',   marker='s', color='red')
axes[1].set_title('Model Loss',  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Per-Language Accuracy Analysis

In [ ]:
df_val = df_train.iloc[TRAIN_SIZE:].copy().reset_index(drop=True)

val_premises    = df_val['premise'].values
val_hypotheses  = df_val['hypothesis'].values
val_preds_raw   = classifier.predict((val_premises, val_hypotheses), batch_size=BATCH_SIZE)
val_preds       = np.argmax(val_preds_raw, axis=1)

df_val['predicted'] = val_preds
df_val['correct']   = (df_val['predicted'] == df_val['label']).astype(int)

lang_acc = df_val.groupby('language')['correct'].mean().sort_values()

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v < 0.7 else '#f39c12' if v < 0.8 else '#27ae60' for v in lang_acc.values]
plt.barh(lang_acc.index, lang_acc.values, color=colors, edgecolor='black')
plt.axvline(x=lang_acc.mean(), color='navy', linestyle='--', label=f'Mean: {lang_acc.mean():.3f}')
plt.xlabel('Accuracy'); plt.title('Per-Language Validation Accuracy', fontsize=14, fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('per_language_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print(lang_acc.to_string())

## 11. Generate Submission

In [ ]:
test_preds_raw  = classifier.predict(
    (df_test['premise'].values, df_test['hypothesis'].values),
    batch_size=BATCH_SIZE
)
test_preds = np.argmax(test_preds_raw, axis=1)

submission = df_test[['id']].copy()
submission['prediction'] = test_preds

submission.to_csv('submission.csv', index=False)
print(f"✅ Submission saved — {len(submission)} rows")
print(submission['prediction'].value_counts().rename(index=LABEL_MAP))
submission.head()

## 12. Save Model for Deployment

In [ ]:
# Save in Keras native format (best for HuggingFace Spaces / Streamlit)
classifier.save(MODEL_SAVE_PATH)
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")

# Verify it can be reloaded
loaded = keras.models.load_model(MODEL_SAVE_PATH)
print("✅ Model reload verified")